In [1]:
import json
import os
import markdown
import pandas as pd

/home/jovyan/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jovyan/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
!pwd

/home/jovyan/LLMOnto/Benchmark/OntologyConceptMatching/data


In [3]:
def load_json(json_path):
    with open(json_path, "rb") as f:
        data = json.load(f)
    return data

def process_df(json_data):
    df = pd.DataFrame(json_data)
    md = df.to_markdown()
    return md

def process_folder(folder_path, onto_name):
    concept_res_folder = folder_path + "concept/"
    concept_res_file_list = os.listdir(concept_res_folder)
    concept_res_file = concept_res_folder + onto_name + "_result.json"
    if os.path.exists(concept_res_file):
        concept_res = load_json(concept_res_file)
        concept_res_md = process_df(concept_res)
    else:
        print("An error occured during handling concept results.")
        concept_res_md = None

    property_res_folder = folder_path + "property/"
    property_res_domain = property_res_folder + onto_name + "_domains.json"
    property_res_property = property_res_folder + onto_name + "_properties.json"
    property_res_range = property_res_folder + onto_name + "_ranges.json"
    
    if os.path.exists(property_res_domain):
        domain_res = load_json(property_res_domain)
        domain_md = process_df(domain_res)
    else:
        print("An error occured during handling property domain results.")
        domain_md = None
        
    if os.path.exists(property_res_property):
        property_res = load_json(property_res_property)
        property_md = process_df(property_res)
    else:
        print("An error occured during handling property results.")
        property_md = None
        
    if os.path.exists(property_res_range):
        range_res = load_json(property_res_range)
        range_md = process_df(range_res)
    else:
        print("An error occured during handling property range results.")
        range_md = None
    
    return concept_res, concept_res_md, domain_res, domain_md, property_res, property_md, range_res, range_md

In [24]:
onto_name = "odrl"
concept_res, concept_res_md, domain_res, domain_md, property_res, property_md, range_res, range_md = process_folder(odrl_path, onto_name)

In [25]:
concept_res

{'hard_match': {'coverage_rate': 0.2962962962962963,
  'precision': 0.5333333333333333,
  'recall': 0.2962962962962963,
  'accuracy': 0.23529411764705882,
  'f1': 0.38095238095238093},
 'sequence_match': {'coverage_rate': 0.6580740740740739,
  'precision': 0.8639922197909068,
  'recall': 0.6580740740740739,
  'accuracy': 0.6580740740740739,
  'f1': 0.7471039629980026},
 'levenshtein': {'coverage_rate': 0.558037037037037,
  'precision': 0.8130699908261833,
  'recall': 0.558037037037037,
  'accuracy': 0.558037037037037,
  'f1': 0.6618347938766994},
 'jaro_winkler': {'coverage_rate': 0.7698518518518517,
  'precision': 0.9224283305227655,
  'recall': 0.7698518518518518,
  'accuracy': 0.7698518518518517,
  'f1': 0.8392619211046957},
 'semantic_embeddinggemma': {'coverage_rate': 0.7122222222222224,
  'precision': 0.9980796179996886,
  'recall': 0.7122222222222223,
  'accuracy': 0.7122222222222224,
  'f1': 0.8312620226079063},
 'semantic_nomic-embed-text': {'coverage_rate': 0.6835555555555554

In [4]:
def generate_row(data, model_name):
    coverage_rate, precision, recall, accuracy, f1 = {"model": model_name}, {"model": model_name}, {"model": model_name}, {"model": model_name}, {"model": model_name}
    for key, item in data.items():
        for sub_key, sub_data in item.items():
            if sub_key == 'coverage_rate':
                coverage_rate[key] = sub_data
            elif sub_key == 'precision':
                precision[key] = sub_data
            elif sub_key == "recall":
                recall[key] = sub_data
            elif sub_key == "accuracy":
                accuracy[key] = sub_data
            elif sub_key == "f1":
                f1[key] = sub_data
            else:
                pass
    coverage_rate_df, precision_df, recall_df, accuracy_df, f1_df = pd.DataFrame([coverage_rate]), pd.DataFrame([precision]), pd.DataFrame([recall]), pd.DataFrame([accuracy]), pd.DataFrame([f1])
    # print(coverage_rate_df)
    coverage_rate_md, precision_md, recall_md, accuracy_md, f1_md = coverage_rate_df.to_markdown(), precision_df.to_markdown(), recall_df.to_markdown(), accuracy_df.to_markdown(), f1_df.to_markdown()
    return coverage_rate_md, precision_md, recall_md, accuracy_md, f1_md

In [5]:
def get_md(data, onto_name, model_name):
    coverage_rate_md, precision_md, recall_md, accuracy_md, f1_md = generate_row(data, model_name)
    row_based_md = f"""
### coverage_rate:
{coverage_rate_md}
### precision:
{precision_md}
### recall:
{recall_md}
### accuracy:
{accuracy_md}
### f1:
{f1_md}
    """
    return row_based_md

In [6]:
def finalize_md(folder_path, onto_name, model_name):
    concept_res, concept_res_md, domain_res, domain_md, property_res, property_md, range_res, range_md = process_folder(folder_path, onto_name)
    concept_md = get_md(concept_res, onto_name, model_name)
    domain_md = get_md(domain_res, onto_name, model_name)
    property_md = get_md(property_res, onto_name, model_name)
    range_md = get_md(range_res, onto_name, model_name)
    final_md = f"""
# Ontology: {onto_name}
## Concept Baseline:
{concept_md}
## Property Baseline:
{property_md}
## Property Domain Baseline:
{domain_md}
## Property Range Baseline:
{range_md}
    """
    return final_md

In [7]:
odrl_neon_path = "/home/jovyan/LLMOnto/Benchmark/OntologyConceptMatching/data/odrl/neon/"
software_neon_path = "/home/jovyan/LLMOnto/Benchmark/OntologyConceptMatching/data/software/neon/"
videogame_neon_path = "/home/jovyan/LLMOnto/Benchmark/OntologyConceptMatching/data/videogame/neon/"

In [8]:
odrl_neon_md = finalize_md(odrl_neon_path, "odrl", "neon-deepseek-reasoner")
software_neon_md = finalize_md(software_neon_path, "software", "neon-deepseek-reasoner")
videogame_neon_md = finalize_md(videogame_neon_path, "videogame", "neon-deepseek-reasoner")

In [10]:
with open('odrl_neon.md', 'w') as f:
    f.write(odrl_neon_md)

In [11]:
with open('software_neon.md', 'w') as f:
    f.write(software_neon_md)

In [12]:
with open('videogame_neon.md', 'w') as f:
    f.write(videogame_neon_md)

In [18]:
import re

In [22]:
def merge_markdown_reports(path_file_1, path_file_2):
    """
    Merges markdown tables from two files with identical structure.
    It takes the headers/structure from file 1 and appends the data rows from file 2.
    """
    try:
        with open(path_file_1, 'r', encoding='utf-8') as f1, \
             open(path_file_2, 'r', encoding='utf-8') as f2:
            content1 = f1.read()
            content2 = f2.read()
    except FileNotFoundError as e:
        return f"Error: {e}"

    # Regex to capture markdown tables (blocks of lines starting with |)
    # This pattern matches contiguous lines that start and end with |
    table_pattern = re.compile(r'(?:^\|.*\|(?:\r?\n|$))+', re.MULTILINE)

    tables1 = list(table_pattern.finditer(content1))
    tables2 = list(table_pattern.finditer(content2))

    if len(tables1) != len(tables2):
        return "Error: Files have a different number of tables and cannot be merged automatically."

    merged_output = []
    last_pos = 0

    # Iterate through both sets of tables
    for match1, match2 in zip(tables1, tables2):
        # Append all text from file 1 that appears before the current table
        merged_output.append(content1[last_pos:match1.start()])

        # Process the tables
        t1_lines = match1.group().strip().splitlines()
        t2_lines = match2.group().strip().splitlines()

        # Markdown tables usually have at least 2 lines (Header + Separator)
        # We append the data rows (index 2 onwards) from the second table
        if len(t2_lines) > 2:
            rows_to_add = t2_lines[2:]
            
            # (Optional) Basic cleanup: If specific index fix is needed, it would go here.
            # Currently, it simply appends the row string.
            
            merged_table = "\n".join(t1_lines + rows_to_add) + "\n"
            merged_output.append(merged_table)
        else:
            # If table 2 is empty/malformed, just keep table 1
            merged_output.append(match1.group())

        last_pos = match1.end()

    # Append the remaining text from file 1
    merged_output.append(content1[last_pos:])

    return "".join(merged_output)

In [25]:
merged_odrl_result = merge_markdown_reports("odrl.md", "odrl_neon.md")
merged_software_result = merge_markdown_reports("software.md", "software_neon.md")
merged_videogame_result = merge_markdown_reports("videogame.md", "videogame_neon.md")

In [26]:
with open('odrl_bench.md', 'w') as f:
    f.write(merged_odrl_result)

In [27]:
with open('software_bench.md', 'w') as f:
    f.write(merged_software_result)

In [28]:
with open('videogame_bench.md', 'w') as f:
    f.write(merged_videogame_result)